# Olist Price Optimizer — Step 1: Data Aggregation

**Goal:** Transform the raw order-item dataset into a clean, monthly, category-level panel  
that downstream notebooks will use to estimate price elasticity and optimise revenue.

**Input:** `data/pricing_analysis_cleaned.parquet` — 109,653 order-item rows (2016-09-15 → 2018-08-29)  
**Output:** `data/category_month_agg.parquet` — one row per (category × month), only categories  
with ≥ 6 months of history (enough observations to fit a reliable price-response curve).

---
### Why aggregate at (category × month)?

* **Category level** — price elasticity requires variation in price across comparable goods.  
  Within a single category prices are set by different sellers, giving us natural experiments.  
  Product-level aggregation would produce too-sparse cells; order-level would add noise.
* **Month granularity** — weekly cells are too sparse for most categories; annual cells destroy  
  the price variation we need. Monthly balances signal richness vs. sample size per cell.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("../data")
print("Libraries loaded. DATA_DIR =", DATA_DIR.resolve())

Libraries loaded. DATA_DIR = C:\Users\lpraz\OneDrive\Área de Trabalho\Masters Degree\02 - Portfolio\03 - Growth\02 - Olist price elasticity\data


---
## 1 — Load Raw Data

The parquet file is the result of joining five Olist public CSVs  
(orders, order_items, products, customers, sellers) and computing  
geolocation features at the order-item grain.

In [2]:
df = pd.read_parquet(DATA_DIR / "pricing_analysis_cleaned.parquet")

print(f"Shape:   {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nDate range:")
print(f"  earliest order: {df['order_purchase_time'].min().date()}")
print(f"  latest order:   {df['order_purchase_time'].max().date()}")
print(f"\nColumn inventory:")
for col in df.columns:
    n_null = df[col].isna().sum()
    null_str = f"  ({n_null:,} nulls)" if n_null else ""
    print(f"  {col:<40s} {str(df[col].dtype):<12s}{null_str}")

Shape:   109,653 rows × 25 columns

Date range:
  earliest order: 2016-09-15
  latest order:   2018-08-29

Column inventory:
  product_id                               object      
  product_category_name                    object      
  product_name_lenght                      float64     
  product_description_lenght               float64     
  product_photos_qty                       float64     
  product_weight_g                         float64     
  product_length_cm                        float64     
  product_height_cm                        float64     
  product_width_cm                         float64     
  price                                    float64     
  freight_value                            float64     
  order_id                                 object      
  customer_id                              object      
  customer_state                           object      
  seller_id                                object      
  seller_state                     

---
## 2 — Engineer `shipping_distance_km` and `delivery_delay_days`

These two features are not in the raw dataset but are needed both as  
cleaning filters and as aggregation signals.

* **`shipping_distance_km`** — great-circle (haversine) distance between  
  customer and seller coordinates. Used to flag geocoding errors (> 4,300 km)  
  and as a logistics-cost proxy in the aggregated panel.
* **`delivery_delay_days`** — actual delivery date minus estimated delivery date.  
  Positive = late. Used to compute a late-delivery rate per (category × month),  
  an operational quality signal correlated with customer satisfaction.

In [3]:
def haversine_km(
    lat1: pd.Series, lon1: pd.Series,
    lat2: pd.Series, lon2: pd.Series,
) -> pd.Series:
    """Return great-circle distance in km between two lat/lon point arrays."""
    R = 6_371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2))
        * np.sin(dlon / 2) ** 2
    )
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


df["shipping_distance_km"] = haversine_km(
    df["customer_lat"], df["customer_lng"],
    df["seller_lat"],   df["seller_lng"],
)

df["delivery_delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

print("shipping_distance_km")
print(df["shipping_distance_km"].describe().round(1).to_string())
print()
print("delivery_delay_days")
print(df["delivery_delay_days"].describe().round(1).to_string())
print(f"\nLate deliveries (delay > 0): {(df['delivery_delay_days'] > 0).sum():,}")

shipping_distance_km
count    109653.0
mean        596.3
std         588.6
min           0.0
25%         185.2
50%         431.9
75%         791.6
max        8677.9

delivery_delay_days
count    109653.0
mean        -12.0
std          10.2
min        -147.0
25%         -17.0
50%         -13.0
75%          -7.0
max         188.0

Late deliveries (delay > 0): 7,218


---
## 3 — Data Cleaning

We remove four types of problematic records before aggregating.  
Each filter targets a specific data-quality issue; keeping them would  
bias the price-response estimates.

| Filter | Reason |
|--------|--------|
| `product_category_name.lower() == 'unknown'` | Catch-all bucket mixing unrelated goods; price signal is meaningless across mixed categories |
| `freight_value == 0` | Data-entry error; inflates avg_price comparisons between categories with different logistics profiles |
| `shipping_distance_km > 4,300` | Brazil's maximum terrestrial extent is ≈ 4,300 km — records beyond this are geocoding artifacts |
| `product_weight_g == 0` | Physically impossible; missing weight recorded as zero, corrupting distance-adjusted pricing features |

In [4]:
n_before = len(df)

mask_unknown  = df["product_category_name"].str.lower() == "unknown"
mask_freight  = df["freight_value"] == 0
mask_distance = df["shipping_distance_km"] > 4_300
mask_weight   = df["product_weight_g"] == 0

drop_mask = mask_unknown | mask_freight | mask_distance | mask_weight
df_clean  = df[~drop_mask].copy()
n_after   = len(df_clean)

print("Records removed per filter (rows may overlap across filters):")
print(f"  unknown category :  {mask_unknown.sum():>6,}")
print(f"  freight == 0     :  {mask_freight.sum():>6,}")
print(f"  distance > 4,300 :  {mask_distance.sum():>6,}")
print(f"  weight == 0      :  {mask_weight.sum():>6,}")
print(f"  ── total unique ──:  {drop_mask.sum():>6,}")
print(f"  rows before      :  {n_before:>6,}")
print(f"  rows after       :  {n_after:>6,}  ({n_after / n_before:.1%} retained)")
print(f"\nCategories remaining: {df_clean['product_category_name'].nunique()}")

Records removed per filter (rows may overlap across filters):


  unknown category :   1,553
  freight == 0     :     380
  distance > 4,300 :       4
  weight == 0      :       8
  ── total unique ──:   1,943
  rows before      :  109,653
  rows after       :  107,710  (98.2% retained)

Categories remaining: 71


---
## 4 — Build `year_month` and Aggregate

We collapse the order-item grain into a **(category × month)** panel.  
Each row in the output dataset represents one category in one calendar month.

**Metrics chosen:**

| Column | Source | Aggregation | Rationale |
|--------|--------|-------------|-----------|
| `sales` | `order_id` | count | Demand proxy: number of items sold |
| `revenue` | `price` | sum | Total revenue the category generated that month |
| `avg_price` | `price` | **median** | Robust central tendency; median suppresses occasional flash-sale outliers |
| `avg_freight` | `freight_value` | **median** | Logistics-cost proxy; affects perceived total price |
| `avg_review` | `review_score` | mean | Customer-satisfaction signal; may moderate price sensitivity |
| `avg_distance` | `shipping_distance_km` | **median** | Logistics complexity; high distance correlates with higher freight |
| `late_delivery_rate` | `delivery_delay_days > 0` | mean | Operational quality; late deliveries erode willingness-to-pay |

We use `pandas.Period` for `year_month` so that time arithmetic is unambiguous.

In [5]:
df_clean["year_month"] = df_clean["order_purchase_time"].dt.to_period("M")

agg = (
    df_clean
    .groupby(["product_category_name", "year_month"])
    .agg(
        sales        =("order_id",             "count"),
        revenue      =("price",                "sum"),
        avg_price    =("price",                "median"),
        avg_freight  =("freight_value",        "median"),
        avg_review   =("review_score",         "mean"),
        avg_distance =("shipping_distance_km", "median"),
    )
    .reset_index()
)

# Late-delivery rate: fraction of items arriving after estimated date
late_rate = (
    df_clean
    .assign(is_late=df_clean["delivery_delay_days"] > 0)
    .groupby(["product_category_name", "year_month"])["is_late"]
    .mean()
    .rename("late_delivery_rate")
    .reset_index()
)
agg = agg.merge(late_rate, on=["product_category_name", "year_month"])

# Month integer (1-12) for seasonality control
agg["month"] = agg["year_month"].dt.month

print(f"Aggregated panel: {agg.shape[0]:,} rows × {agg.shape[1]} columns")
print(f"Categories:       {agg['product_category_name'].nunique()}")
print(f"Months covered:   {agg['year_month'].nunique()}")
print()
print(agg.dtypes.to_string())
print()
print(agg.head(3).to_string(index=False))

Aggregated panel: 1,243 rows × 10 columns
Categories:       71
Months covered:   23

product_category_name       object
year_month               period[M]
sales                        int64
revenue                    float64
avg_price                  float64
avg_freight                float64
avg_review                 float64
avg_distance               float64
late_delivery_rate         float64
month                        int64

     product_category_name year_month  sales  revenue  avg_price  avg_freight  avg_review  avg_distance  late_delivery_rate  month
agro_industry_and_commerce    2017-01      3    65.97     21.990         8.72    4.333333     18.912869                 0.0      1
agro_industry_and_commerce    2017-02      7   224.84     21.990        14.52    3.857143    443.827882                 0.0      2
agro_industry_and_commerce    2017-03      2    81.99     40.995        14.35    2.000000    318.790690                 0.0      3


---
## 5 — Validate: Category Coverage

A price-response curve estimated from fewer than **6 data points** (months)  
is statistically unreliable — the polynomial can fit the noise rather than  
the signal. We therefore keep only categories with ≥ 6 months of history.

Below we show:
1. How many categories pass the threshold.
2. The top 15 categories by total sales volume (with their month counts).

In [6]:
months_per_cat = (
    agg.groupby("product_category_name")["year_month"]
    .nunique()
    .rename("n_months")
    .sort_values(ascending=False)
)

n_pass = (months_per_cat >= 6).sum()
n_fail = (months_per_cat <  6).sum()
print(f"Categories with >= 6 months: {n_pass} / {len(months_per_cat)}")
print(f"Categories with  < 6 months: {n_fail}  (will be dropped)")
print()

# Top 15 by total sales
top15_sales = (
    agg.groupby("product_category_name")["sales"]
    .sum()
    .rename("total_sales")
)
summary = pd.DataFrame({"total_sales": top15_sales, "n_months": months_per_cat})
summary = summary.sort_values("total_sales", ascending=False).head(15)
summary["qualifies"] = summary["n_months"] >= 6

print("Top 15 categories by total sales:")
print(f"  {'Category':<40s} {'Total Sales':>12s}  {'Months':>6s}  {'>=6?':>5s}")
print("  " + "-" * 70)
for cat, row in summary.iterrows():
    tick = "yes" if row["qualifies"] else "NO "
    print(f"  {cat:<40s} {int(row['total_sales']):>12,}  {int(row['n_months']):>6}  {tick:>5s}")

Categories with >= 6 months: 70 / 71
Categories with  < 6 months: 1  (will be dropped)

Top 15 categories by total sales:
  Category                                  Total Sales  Months   >=6?
  ----------------------------------------------------------------------
  bed_bath_table                                 10,909      21    yes
  health_beauty                                   9,435      22    yes
  sports_leisure                                  8,414      21    yes
  furniture_decor                                 8,044      21    yes
  computers_accessories                           7,622      21    yes
  housewares                                      6,777      21    yes
  watches_gifts                                   5,622      21    yes
  telephony                                       4,402      21    yes
  garden_tools                                    4,201      21    yes
  auto                                            4,128      21    yes
  toys                  

---
## 6 — Filter to Qualified Categories

We drop any category with fewer than 6 months of sales data.  
These are typically niche products with very few transactions — not enough  
to distinguish a price trend from random fluctuation.

In [7]:
qualified_cats = months_per_cat[months_per_cat >= 6].index
agg_filtered = agg[agg["product_category_name"].isin(qualified_cats)].copy()

print(f"Categories retained: {agg_filtered['product_category_name'].nunique()}")
print(f"Rows retained:       {len(agg_filtered):,} (of {len(agg):,})")
print()
print("Month distribution across the panel:")
print(agg_filtered["year_month"].value_counts().sort_index().to_string())

Categories retained: 70
Rows retained:       1,241 (of 1,243)

Month distribution across the panel:
year_month
2016-09     1
2016-10    29
2016-12     1
2017-01    41
2017-02    50
2017-03    52
2017-04    56
2017-05    58
2017-06    59
2017-07    61
2017-08    62
2017-09    64
2017-10    63
2017-11    65
2017-12    62
2018-01    66
2018-02    63
2018-03    66
2018-04    68
2018-05    63
2018-06    64
2018-07    64
2018-08    63
Freq: M


---
## 7 — Save to Parquet

We convert `year_month` from `Period` to `str` (e.g. `"2017-03"`) before  
writing, because the Parquet format does not natively support pandas Period.  
Downstream notebooks will parse it back with `pd.PeriodIndex(..., freq='M')`.

In [8]:
out_path = DATA_DIR / "category_month_agg.parquet"

agg_out = agg_filtered.copy()
agg_out["year_month"] = agg_out["year_month"].astype(str)

agg_out.to_parquet(out_path, index=False)
print(f"Saved: {out_path.resolve()}")
print(f"Shape: {agg_out.shape}")
print()
print("Final column dtypes:")
print(agg_out.dtypes.to_string())
print()
print("Sample (first 5 rows):")
print(agg_out.head().to_string(index=False))

Saved: C:\Users\lpraz\OneDrive\Área de Trabalho\Masters Degree\02 - Portfolio\03 - Growth\02 - Olist price elasticity\data\category_month_agg.parquet
Shape: (1241, 10)

Final column dtypes:
product_category_name     object
year_month                object
sales                      int64
revenue                  float64
avg_price                float64
avg_freight              float64
avg_review               float64
avg_distance             float64
late_delivery_rate       float64
month                      int64

Sample (first 5 rows):
     product_category_name year_month  sales  revenue  avg_price  avg_freight  avg_review  avg_distance  late_delivery_rate  month
agro_industry_and_commerce    2017-01      3    65.97     21.990         8.72    4.333333     18.912869                 0.0      1
agro_industry_and_commerce    2017-02      7   224.84     21.990        14.52    3.857143    443.827882                 0.0      2
agro_industry_and_commerce    2017-03      2    81.99     40.99